In [2]:
import pandas as pd
import numpy as np
import scanpy as sc
import os,gc

sc.settings.verbosity = 2

data_dir = "biomni_dup/onc_promoter/data/rna_137398"

In [3]:
timepoint_map = {
    "GSE137398_ONCRGCs_control_count_mat.csv.gz": "ctrl",
    "GSE137398_ONCRGCs_12h_afterCrush_count_mat.csv.gz": "12h",
    "GSE137398_ONCRGCs_1d_afterCrush_count_mat.csv.gz": "1d",
    "GSE137398_ONCRGCs_2d_afterCrush_count_mat.csv.gz": "2d",
    "GSE137398_ONCRGCs_4d_afterCrush_count_mat.csv.gz": "4d",
    "GSE137398_ONCRGCs_1w_afterCrush_count_mat.csv.gz": "1w",
    "GSE137398_ONCRGCs_2w_afterCrush_count_mat.csv.gz": "2w",
}

In [ ]:

test_file = "GSE137398_ONCRGCs_control_count_mat.csv.gz"
df_test = pd.read_csv(os.path.join(data_dir, test_file), index_col=0, nrows=5)


In [ ]:
print("=== First file inspection ===")
print(f"Shape (first 5 rows): {df_test.shape}")
print(f"Index name: {df_test.index.name}")
print(f"Index sample: {list(df_test.index[:5])}")
print(f"Columns sample: {list(df_test.columns[:5])}")
print(f"\nFirst 5 rows, first 5 cols:")
print(df_test.iloc[:5, :5])
print(f"\nDtypes: {df_test.dtypes.unique()}")


=== First file inspection ===
Shape (first 5 rows): (5, 17887)
Index name: None
Index sample: ['4933401J01Rik', 'Gm26206', 'Gm1992', 'Gm10568', 'Gm38385']
Columns sample: ['CtC57CD45CD90P1_AAACCTGAGCTAACAA-1', 'CtC57CD45CD90P1_AAACCTGCACGTAAGG-1', 'CtC57CD45CD90P1_AAACCTGCAGACAAAT-1', 'CtC57CD45CD90P1_AAACCTGTCCGCGGTA-1', 'CtC57CD45CD90P1_AAACGGGAGCTCTCGG-1']

First 5 rows, first 5 cols:
               CtC57CD45CD90P1_AAACCTGAGCTAACAA-1  \
4933401J01Rik                                   0   
Gm26206                                         0   
Gm1992                                          0   
Gm10568                                         0   
Gm38385                                         0   

               CtC57CD45CD90P1_AAACCTGCACGTAAGG-1  \
4933401J01Rik                                   0   
Gm26206                                         0   
Gm1992                                          0   
Gm10568                                         0   
Gm38385                  

In [ ]:
#Memory intensive step - make sure you have at least 16 GB RAM
adatas = {}
for fname, tp in timepoint_map.items():
    print(f"Loading {tp}...", flush=True)
    df = pd.read_csv(os.path.join(data_dir, fname), index_col=0)
    # genes are rows, cells are columns -> transpose for AnnData (cells x genes)
    adata = sc.AnnData(X=df.T.values.astype(np.float32),
                       obs=pd.DataFrame(index=df.columns),
                       var=pd.DataFrame(index=df.index))
    adata.obs['timepoint'] = tp
    adata.var_names = df.index.tolist()
    adata.obs_names = df.columns.tolist()
    # Make unique
    adata.var_names_make_unique()
    adata.obs_names_make_unique()
    adatas[tp] = adata
    print(f"  {tp}: {adata.shape[0]} cells x {adata.shape[1]} genes")
    del df
    gc.collect()

Loading ctrl...
  ctrl: 17887 cells x 40790 genes
Loading 12h...
  12h: 14720 cells x 40790 genes
Loading 1d...
  1d: 14627 cells x 40790 genes
Loading 2d...
  2d: 13906 cells x 40790 genes
Loading 4d...
  4d: 17216 cells x 40790 genes
Loading 1w...
  1w: 13701 cells x 40790 genes
Loading 2w...
  2w: 14189 cells x 40790 genes


In [ ]:
adata_all = sc.concat(list(adatas.values()), join='outer', merge='same')
adata_all.obs['timepoint'] = pd.Categorical(
    adata_all.obs['timepoint'],
    categories=['ctrl', '12h', '1d', '2d', '4d', '1w', '2w'],
    ordered=True
)
print(f"\n=== Combined dataset ===")
print(f"Total: {adata_all.shape[0]} cells x {adata_all.shape[1]} genes")
print(f"Timepoint distribution:\n{adata_all.obs['timepoint'].value_counts().sort_index()}")



=== Combined dataset ===
Total: 106246 cells x 40790 genes
Timepoint distribution:
timepoint
ctrl    17887
12h     14720
1d      14627
2d      13906
4d      17216
1w      13701
2w      14189
Name: count, dtype: int64


In [ ]:
#dsaving raw data
adata_all.write_h5ad("biomni_dup/onc_promoter/data/adata_combined_raw.h5ad")